# ARC 1.0 — agent model router on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/ARC-1.0-agent-model-router.ipynb)

Run Rayline's [`rayline-ai/ARC-1.0`](https://huggingface.co/rayline-ai/ARC-1.0) locally: give it an agent conversation and get **per-model success probability and token-usage predictions**, then combine those with **your prices** using the card's utility policy. The same backbone also scores **PII presence**, **request categories**, and **developer-defined questions**.

ARC — the Adaptive Routing Controller — is a **0.6B Apache-2.0 decision model**, a fine-tune of [`Qwen/Qwen3.5-0.8B`](https://huggingface.co/Qwen/Qwen3.5-0.8B) (`base_model_relation: finetune`). It is **not a chat model**: it does not write the next assistant turn. It predicts routing signals; the application supplies the model pool, list prices, and the value of success.

See [use cases](https://github.com/donvito/notebooks/blob/main/docs/ARC-1.0-agent-model-router.md) for when this notebook is a good fit.

**Why this model.** Most routers either call a hosted API or emit text that you then parse. ARC **separates predictions from policy**. The [card](https://huggingface.co/rayline-ai/ARC-1.0) states the policy as:

**Utility = V × predicted success − expected cost**

| Preset | V (USD) | Behaviour |
| --- | --- | --- |
| **ARC Sage** | 4.5 | Places greater value on successful completion. |
| **ARC Swift** | 0.1 | Places greater weight on reducing cost. |

Updating prices does not require retraining the predictors. The question head is trained primarily for routing decisions (*Does this request need a stronger model? Which of these models should answer? Will the proposed model succeed?*).

**How this notebook loads it (T4 path).** The Hub `library_name` is **vLLM** and the architecture is custom `ArcImagePolicyForPooling`. There is **no** `modeling_*.py`, so `transformers` + `trust_remote_code` cannot load that class. Stock vLLM cannot either. The public [`vllm-arc`](https://github.com/rayline-ai/vllm-arc) plugin is a work in progress: Stage-1 scoring only, on a CUDA 13 vLLM **fork** that Colab T4 images do not ship. The hosted Routes API needs a Rayline key — this notebook uses **no secrets**.

The release [NOTICE](https://huggingface.co/rayline-ai/ARC-1.0/blob/main/NOTICE) says text layers **0–11 are unchanged** from `Qwen/Qwen3.5-0.8B`, with specialized heads on top. This notebook therefore:

1. Loads **`Qwen/Qwen3.5-0.8B`** in **float16 + SDPA/eager** (T4 has no native bf16 and no FlashAttention 2).
2. Reads last-token hidden states after layer 12 (`block_11` in the card) and applies the released **Stage-1** head (`vllm_arc.arc_heads`, Apache-2.0) plus the **PII / topic** MLPs (`signals.*`).
3. Overlays the released **question-head** upper layers (12–23) and bilinear pointer (`question_head/qh4in-full-e3/`), using the rendering in that addon's `manifest.json`.

Stage-2 (per-model **reasoning-effort** arms) is in the weights; the public plugin does **not** serve it yet, so the demos apply the card's utility policy on Stage-1 success and usage. The GGUF under `gguf/` is the encoder for [Rayline CLI local mode](https://docs.rayline.ai) (preview), not this notebook.

**Licence:** Apache 2.0 (ARC 1.0, Qwen3.5-0.8B, vllm-arc). PII training used [Nemotron-PII](https://huggingface.co/datasets/nvidia/Nemotron-PII) (CC-BY-4.0); that dataset is not redistributed here.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about **15 GB** VRAM, compute capability **7.5**. Hub files used here: Qwen3.5-0.8B **~1.7 GB**, ARC `model.safetensors` **1.3 GB** (heads only; encoder tensors skipped in RAM), question-head upper layers **~1.0 GB**. Combined download is about **4 GB**. Expected peak VRAM on these short text demos is about **2–5 GB** — an estimate from the 0.8B float16 backbone plus a short window, not a measured Colab run.

**Qwen3.5 linear-attention.** Do **not** install `flash-attn`, `flash-linear-attention`, or `causal-conv1d`. Transformers already has a slower PyTorch fallback.

**Links:** [model card](https://huggingface.co/rayline-ai/ARC-1.0) · [docs](https://docs.rayline.ai) · [technical report](https://static.rayline.ai/rayline/reports/arc/latest) · [vllm-arc](https://github.com/rayline-ai/vllm-arc) · [Rayline CLI](https://github.com/rayline-ai/rayline-cli) · [announcement](https://x.com/davidvgilmore/status/2120583960447029699)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. This cell imports **torch only** so the later transformers pin can take effect. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

DTYPE = torch.float16 if major < 8 else torch.bfloat16
BACKBONE_ID = "Qwen/Qwen3.5-0.8B"
RELEASE_ID = "rayline-ai/ARC-1.0"

free_b, total_b = torch.cuda.mem_get_info()
print("torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free_b / 1024**3:.2f} GB of {total_b / 1024**3:.1f} GB")
print("dtype:", DTYPE, "| native bf16:", NATIVE_BF16)
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("T4 path: float16 + SDPA/eager. bfloat16 and FlashAttention-2 are off.")


## 2. Install Transformers and the Stage-1 head

The [card](https://huggingface.co/rayline-ai/ARC-1.0) does not ship a Transformers `auto_map`. Qwen3.5 needs a current Transformers; this notebook pins **5.20.1** (same pin as other Qwen3.5 T4 notebooks here). [`vllm-arc`](https://github.com/rayline-ai/vllm-arc) is installed **without vLLM** — `vllm_arc.arc_heads` is torch-only and runs the compiled Stage-1 tensors.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. Do **not** install `flash-attn`, `flash-linear-attention`, `causal-conv1d`, or `vllm`.


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate>=1.1.0" "tokenizers>=0.22" "safetensors>=0.7" huggingface_hub
%pip install -q "git+https://github.com/rayline-ai/vllm-arc"

import sys

# Drop a pre-imported Colab transformers so 5.20.1 is visible without Restart session.
for name in list(sys.modules):
    if name == "transformers" or name.startswith("transformers."):
        del sys.modules[name]

import transformers
from packaging.version import Version

assert Version(transformers.__version__.split("+")[0]) >= Version("5.14.1"), (
    f"transformers {transformers.__version__} is loaded; ARC 1.0 needs Qwen3_5 support. "
    "Runtime > Restart session, then rerun from the GPU check cell."
)
print("transformers", transformers.__version__)
import vllm_arc.arc_heads as arc_heads

print("vllm_arc.arc_heads", arc_heads.__file__)


> If the version assert failed, use *Runtime > Restart session* and continue from the GPU-check cell (the pin is already installed).


## 3. Download the ARC release and load the backbone

Snapshot **config, tokenizer, Stage-1/signal weights, prices, and the question-head addon**. Skip the 1.0 GB GGUF (Rayline CLI local mode) and the goldens. Then load `Qwen/Qwen3.5-0.8B` and overlay `question_head/qh4in-full-e3/upper.safetensors` on layers 12–23.

Heads are read with `safetensors.safe_open` so the 1.3 GB file is **not** copied into RAM in full (`vllm_arc.heads.load_stage_one` would `load_file` every tensor, including the encoder).


In [ ]:
import gc
import json
from pathlib import Path

from huggingface_hub import snapshot_download
from safetensors import safe_open
from safetensors.torch import load_file
from transformers import AutoConfig, AutoTokenizer
from vllm_arc.arc_heads import stage_one_module
from vllm_arc.heads import StageOneHead

RELEASE_DIR = Path(
    snapshot_download(
        RELEASE_ID,
        allow_patterns=[
            "arc_release.json",
            "config.json",
            "model.safetensors",
            "presets.json",
            "tokenizer.json",
            "tokenizer_config.json",
            "chat_template.jinja",
            "runtime/prices.json",
            "vocab.json",
            "merges.txt",
            "question_head/qh4in-full-e3/*",
        ],
    )
)
print("release dir:", RELEASE_DIR)

ARC_CFG = json.loads((RELEASE_DIR / "config.json").read_text())
PRESETS = json.loads((RELEASE_DIR / "presets.json").read_text())
PRICES = json.loads((RELEASE_DIR / "runtime" / "prices.json").read_text())["models"]
QH_MANIFEST = json.loads(
    (RELEASE_DIR / "question_head" / "qh4in-full-e3" / "manifest.json").read_text()
)
STAGE_ONE_META = ARC_CFG["arc"]["stage_one"]
SIGNAL_META = ARC_CFG["arc"]["signals"]["heads"]

print("release:", ARC_CFG["arc"]["release"], "| depth:", ARC_CFG["arc"]["depth"])
print("architectures:", ARC_CFG.get("architectures"))
print("sage V:", PRESETS["presets"]["sage"]["value_usd"], "| swift V:", PRESETS["presets"]["swift"]["value_usd"])
print("stage-one candidates:", len(STAGE_ONE_META["candidates"]), "| regimes:", STAGE_ONE_META["regimes"])
print("signal heads:", list(SIGNAL_META))
print("question head:", QH_MANIFEST["name"], QH_MANIFEST["pointer_head"]["kind"])


def print_vram(label=""):
    suffix = f"  ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
        f"{suffix}"
    )


def text_model(module):
    inner = getattr(module, "model", module)
    if hasattr(inner, "language_model"):
        return inner.language_model
    if hasattr(inner, "layers"):
        return inner
    if hasattr(module, "language_model"):
        return module.language_model
    raise RuntimeError(f"could not find the Qwen3.5 text stack on {type(module)}")


fast_ok = False
try:
    from transformers.utils import is_causal_conv1d_available, is_flash_linear_attention_available

    fast_ok = bool(is_flash_linear_attention_available() and is_causal_conv1d_available())
except Exception:
    fast_ok = False
print("linear-attention fast path (flash-linear-attention + causal-conv1d):", fast_ok)
if not fast_ok:
    print("using the Transformers PyTorch fallback for Qwen3.5 linear-attention layers")

hf_cfg = AutoConfig.from_pretrained(BACKBONE_ID)
print("backbone architectures:", getattr(hf_cfg, "architectures", None), "| model_type:", hf_cfg.model_type)

LOADER_NAMES = (
    "AutoModelForImageTextToText",
    "AutoModelForMultimodalLM",
    "AutoModelForCausalLM",
    "AutoModel",
)


def iter_loaders():
    import transformers as tf

    for name in LOADER_NAMES:
        cls = getattr(tf, name, None)
        if cls is not None:
            yield name, cls


def load_backbone(attn):
    kwargs = dict(dtype=DTYPE, device_map="auto")
    if attn is not None:
        kwargs["attn_implementation"] = attn
    last_error = None
    for name, cls in iter_loaders():
        try:
            print(f"loading {BACKBONE_ID} with {name} attn={attn}")
            return cls.from_pretrained(BACKBONE_ID, **kwargs).eval(), name
        except Exception as exc:
            last_error = exc
            print(f"{name} refused:", exc)
    raise RuntimeError(f"could not load {BACKBONE_ID}") from last_error


torch.cuda.reset_peak_memory_stats()
model = None
ATTN = None
loader_name = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model, loader_name = load_backbone(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()
if model is None:
    raise RuntimeError(f"could not load {BACKBONE_ID}") from last_error

TEXT = text_model(model)
print("loader:", loader_name, "| class:", type(model).__name__, "| text:", type(TEXT).__name__, "| attn:", ATTN)
if not hasattr(TEXT, "norm"):
    raise RuntimeError(f"{type(TEXT).__name__} has no .norm — cannot apply the card's RMSNorm readout")

upper_path = RELEASE_DIR / "question_head" / "qh4in-full-e3" / "upper.safetensors"
upper = load_file(str(upper_path))
current = TEXT.state_dict()
mapped = {}
unmapped = []
for key, value in upper.items():
    dest = None
    for candidate in (key, f"model.{key}", key.removeprefix("model.")):
        if candidate in current:
            dest = candidate
            break
    if dest is None:
        unmapped.append(key)
        continue
    mapped[dest] = value.to(device=current[dest].device, dtype=current[dest].dtype)
missing = TEXT.load_state_dict(mapped, strict=False)
print(f"overlaid {len(mapped)} / {len(upper)} question-head upper tensors")
if unmapped:
    print("unmapped overlay keys (first 8):", unmapped[:8])
if len(mapped) < 50:
    raise RuntimeError(
        f"question-head overlay mapped {len(mapped)} tensors; expected the 12 fine-tuned upper layers. "
        f"TEXT keys look like: {list(current)[:8]}"
    )
if missing.unexpected_keys:
    print("unexpected overlay keys (first 8):", missing.unexpected_keys[:8])

n_params = sum(p.numel() for p in TEXT.parameters())
print(f"text parameters: {n_params / 1e9:.3f}B | dtype: {next(TEXT.parameters()).dtype}")
print_vram("after backbone load")


Load Stage-1 (success + four-leg token usage) and the three signal MLPs from `model.safetensors`. Tokenizer comes from the ARC snapshot (same files as the Qwen3.5-0.8B revision named in NOTICE).


In [ ]:
TOKENIZER = AutoTokenizer.from_pretrained(RELEASE_DIR)

stage_tensors = {}
signal_tensors = {}
with safe_open(str(RELEASE_DIR / "model.safetensors"), framework="pt") as handle:
    for key in handle.keys():
        if key.startswith("stage_one.") and key != "stage_one.auxiliary":
            stage_tensors[key.removeprefix("stage_one.")] = handle.get_tensor(key)
        elif key.startswith("signals."):
            signal_tensors[key] = handle.get_tensor(key)

device = torch.device(DEVICE)
STAGE_ONE = StageOneHead(
    module=stage_one_module({k: v.to(device) for k, v in stage_tensors.items() if k != "auxiliary"}),
    release=ARC_CFG["arc"]["release"],
    selector_id=STAGE_ONE_META["selector_id"],
    candidates=list(STAGE_ONE_META["candidates"]),
    regimes=list(STAGE_ONE_META["regimes"]),
    reads_context="ctx_mean" in stage_tensors,
    device=device,
    reads_cache="w_cache" in stage_tensors,
)
print(
    "stage-one",
    STAGE_ONE.release,
    "| reads_context:",
    STAGE_ONE.reads_context,
    "| reads_cache:",
    STAGE_ONE.reads_cache,
    "| n tensors:",
    len(stage_tensors),
)

pointer_path = RELEASE_DIR / "question_head" / "qh4in-full-e3" / "pointer.safetensors"
with safe_open(str(pointer_path), framework="pt") as handle:
    POINTER = {key: handle.get_tensor(key).to(device=device, dtype=torch.float32) for key in handle.keys()}
print("pointer tensors:", list(POINTER), "| logits:", QH_MANIFEST["pointer_head"]["logits"])

WIRE = {
    "claude-haiku-5.5": "anthropic/claude-haiku-5.5",
    "claude-opus-5.5": "anthropic/claude-opus-5.5",
    "claude-sonnet-5.5": "anthropic/claude-sonnet-5.5",
    "deepseek-v4.1-flash": "deepseek/deepseek-v4.1-flash",
    "glm-5.3": "z-ai/glm-5.3",
    "glm-5.3-flash": "z-ai/glm-5.3-flash",
    "gpt-5.6-terra": "openai/gpt-5.6-terra",
    "gpt-6-astra": "openai/gpt-6-astra",
    "gpt-6-luna": "openai/gpt-6-luna",
    "gpt-6.1-sol": "openai/gpt-6.1-sol",
    "kimi-k3": "moonshotai/kimi-k3",
}
NAMED_INDEX = {
    cand["model"]: i
    for i, cand in enumerate(STAGE_ONE.candidates)
    if cand.get("model") in WIRE
}
if not NAMED_INDEX:
    raise RuntimeError("no named Stage-1 candidates intersect runtime/prices.json")
print("named pool:", list(NAMED_INDEX))
print_vram("after heads")


## 4. Helpers

Role-block rendering follows the question-head addon's `manifest.json` (`<|im_start|>{role}\n...\n<|im_end|>\n`). Routing/PII readouts are **last-token hidden states after layer 12**, then the backbone's final RMSNorm (`block_11` / `norm(h_12)` in the release). Question-head readouts are **last_hidden_state** after all 24 layers.

Token prices in `runtime/prices.json` are a **reference card** of list rates (uncached input, cache read, cache write, output). Callers override them; the notebook uses that card so Sage vs Swift is visible. Expected cost is `sum(usage_leg × $/MTok) / 1e6`.


In [ ]:
USAGE_LEGS = ("uncached_in", "cache_read", "cache_write", "output")
SAGE_V = float(PRESETS["presets"]["sage"]["value_usd"])
SWIFT_V = float(PRESETS["presets"]["swift"]["value_usd"])
PLATT_EPS = 1e-6
ENCODER_LAYER = 12  # hidden_states[12] == output of layer 11 (depth-12 encoder)


def message_text(content):
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(item.get("text") or "")
        return "".join(parts)
    return str(content)


def render_role_blocks(messages):
    parts = []
    for message in messages:
        role = message["role"]
        text = message_text(message.get("content")).rstrip()
        parts.append(f"<|im_start|>{role}\n{text}<|im_end|>\n")
    return "".join(parts)


def encode_ids(text):
    return TOKENIZER(text, add_special_tokens=False, return_tensors="pt")


@torch.no_grad()
def forward_text(input_ids):
    input_ids = input_ids.to(DEVICE)
    mask = torch.ones_like(input_ids)
    return TEXT(
        input_ids=input_ids,
        attention_mask=mask,
        output_hidden_states=True,
        use_cache=False,
    )


def last_token(hidden, input_ids):
    # hidden: [batch, seq, dim]; last non-pad token (these demos have no pad).
    return hidden[0, input_ids.shape[-1] - 1, :]


@torch.no_grad()
def encoder_row(messages):
    ids = encode_ids(render_role_blocks(messages))["input_ids"]
    out = forward_text(ids)
    hidden_states = out.hidden_states
    if hidden_states is None or len(hidden_states) <= ENCODER_LAYER:
        raise RuntimeError(
            f"need hidden_states[{ENCODER_LAYER}] (block_11); got "
            f"{None if hidden_states is None else len(hidden_states)} tensors"
        )
    h12 = hidden_states[ENCODER_LAYER]
    row = TEXT.norm(last_token(h12, ids))
    return row.float(), int(ids.shape[-1])


def platt_map(p, ab):
    q = p.clamp(PLATT_EPS, 1 - PLATT_EPS)
    a, b = ab[..., 0], ab[..., 1]
    z = a * torch.log(q / (1 - q)) + b
    return torch.sigmoid(z)


def signal_probs(row, prefix, class_names):
    mean = signal_tensors[f"{prefix}.mean"].to(device=DEVICE, dtype=torch.float32)
    scale = signal_tensors[f"{prefix}.scale"].to(device=DEVICE, dtype=torch.float32)
    w0 = signal_tensors[f"{prefix}.net.0.weight"].to(device=DEVICE, dtype=torch.float32)
    b0 = signal_tensors[f"{prefix}.net.0.bias"].to(device=DEVICE, dtype=torch.float32)
    w2 = signal_tensors[f"{prefix}.net.2.weight"].to(device=DEVICE, dtype=torch.float32)
    b2 = signal_tensors[f"{prefix}.net.2.bias"].to(device=DEVICE, dtype=torch.float32)
    ab = signal_tensors[f"{prefix}.platt"].to(device=DEVICE, dtype=torch.float32)
    z = (row.to(torch.float32) - mean) / scale.clamp_min(1e-6)
    hidden = torch.relu(z @ w0.T + b0)
    logits = hidden @ w2.T + b2
    raw = torch.sigmoid(logits)
    mapped = platt_map(raw, ab)
    return {name: float(mapped[i]) for i, name in enumerate(class_names)}


def bucket_threshold(n_tokens, head_cfg):
    chosen = head_cfg["length_buckets"][0]
    for bucket in head_cfg["length_buckets"]:
        chosen = bucket
        if n_tokens <= bucket["max_tokens"]:
            break
    return chosen


def expected_cost_usd(usage_4, wire_id):
    rates = PRICES[wire_id]  # $/MTok: uncached, cache read, cache write, output
    return sum(float(tok) * float(rate) / 1e6 for tok, rate in zip(usage_4, rates))


def utility(success, cost, value_usd):
    return value_usd * success - cost


@torch.no_grad()
def route(messages, pool=None):
    row, n_tokens = encoder_row(messages)
    kwargs = {}
    if STAGE_ONE.reads_context:
        width = int(STAGE_ONE.module.ctx_mean.shape[0])
        context = torch.zeros(1, width, dtype=torch.float64, device=device)
        context[0, 0] = torch.log1p(torch.tensor(float(n_tokens), dtype=torch.float64))
        kwargs["context"] = context
    p, usage = STAGE_ONE(row.to(torch.float64).unsqueeze(0), "fixed", **kwargs)
    point_p = p[0, 0]
    point_u = usage[0, 0]
    pool = pool or NAMED_INDEX
    rows = []
    for name, index in pool.items():
        success = float(point_p[index])
        if success != success:  # NaN: withheld / not served
            continue
        legs = [float(point_u[index, k]) for k in range(4)]
        cost = expected_cost_usd(legs, WIRE[name])
        rows.append(
            {
                "model": name,
                "wire": WIRE[name],
                "success": success,
                "usage": dict(zip(USAGE_LEGS, legs)),
                "cost_usd": cost,
                "sage": utility(success, cost, SAGE_V),
                "swift": utility(success, cost, SWIFT_V),
            }
        )
    rows.sort(key=lambda r: -r["sage"])
    return {"n_tokens": n_tokens, "rows": rows}


def pick(rows, key):
    return max(rows, key=lambda r: r[key])


def print_route_table(result, title):
    print(f"\n=== {title} ===")
    print(f"encoded tokens: {result['n_tokens']}")
    print(
        f"{'model':<22} {'P(success)':>10} {'uncached':>9} {'out':>8} "
        f"{'cost$':>9} {'Sage U':>9} {'Swift U':>9}"
    )
    for row in result["rows"]:
        print(
            f"{row['model']:<22} {row['success']:10.3f} {row['usage']['uncached_in']:9.0f} "
            f"{row['usage']['output']:8.0f} {row['cost_usd']:9.4f} {row['sage']:9.4f} {row['swift']:9.4f}"
        )
    sage = pick(result["rows"], "sage")
    swift = pick(result["rows"], "swift")
    print(f"Sage  (V={SAGE_V}) picks {sage['model']}  (U={sage['sage']:.4f})")
    print(f"Swift (V={SWIFT_V}) picks {swift['model']}  (U={swift['swift']:.4f})")
    if sage["model"] != swift["model"]:
        print("Sage and Swift disagree — the same predictions, different V.")
    else:
        print("Sage and Swift agree on this pool for this request.")


print("helpers ready | Sage V", SAGE_V, "| Swift V", SWIFT_V)


## 5. Demo 1 — route sample agent conversations

Three short conversations, one Stage-1 forward each. The pool is the **named** models that ship both a Stage-1 candidate and a row in `runtime/prices.json` (withheld candidates are skipped). Prices are the release's reference card, not anyone's production rate.


In [ ]:
CONVERSATIONS = [
    {
        "name": "easy arithmetic",
        "messages": [{"role": "user", "content": "What is two plus two?"}],
    },
    {
        "name": "short rewrite",
        "messages": [
            {
                "role": "user",
                "content": "Rewrite this sentence so it is friendlier: Your invoice is overdue.",
            }
        ],
    },
    {
        "name": "coding agent",
        "messages": [
            {
                "role": "user",
                "content": (
                    "Our checkout API returns 500 when the cart has a discount plus a gift card. "
                    "Trace the Stripe webhook handler, add a failing test, and patch it. "
                    "Keep the public API stable."
                ),
            },
            {
                "role": "assistant",
                "content": "I'll inspect the webhook handler and the discount stack next.",
            },
            {
                "role": "user",
                "content": (
                    "The retry queue also double-charges if Stripe sends the same event twice. "
                    "Fix idempotency while you are in there, then open a PR description."
                ),
            },
        ],
    },
]

ROUTES = []
for case in CONVERSATIONS:
    result = route(case["messages"])
    ROUTES.append((case, result))
    print_route_table(result, case["name"])

print_vram("demo 1")


## 6. Demo 2 — PII detection and request categorization

The release ships three multilabel heads on the **same** depth-12 readout (`signals.pii.payload`, `signals.topic.domain`, `signals.topic.operation`). Architecture from the tensors: standardize → Linear(1024, 256) → ReLU → Linear(256, C) → sigmoid → Platt map stored in `*.platt`. Thresholds come from `config.json` (PII `any` uses the length-bucket threshold; per-type PII uses `type_threshold` 0.5; topics use `class_thresholds`).

The PII sample is **fictional** (`example.com`, `+1-555-0100`). Do not paste real user data into Colab.


In [ ]:
PII_CLASSES = SIGNAL_META["pii.payload"]["classes"]
DOMAIN_CLASSES = SIGNAL_META["topic.domain"]["classes"]
OPERATION_CLASSES = SIGNAL_META["topic.operation"]["classes"]
PII_TYPE_THRESHOLD = float(SIGNAL_META["pii.payload"]["type_threshold"])


def present(probs, thresholds):
    hits = [name for name, p in probs.items() if p >= thresholds[name]]
    top = max(probs, key=probs.get)
    return hits, top


def report_signals(title, messages):
    row, n_tokens = encoder_row(messages)
    pii = signal_probs(row, "signals.pii.payload", PII_CLASSES)
    domain = signal_probs(row, "signals.topic.domain", DOMAIN_CLASSES)
    operation = signal_probs(row, "signals.topic.operation", OPERATION_CLASSES)
    bucket = bucket_threshold(n_tokens, SIGNAL_META["pii.payload"])
    pii_any_t = float(bucket["threshold"])
    pii_hits = [
        name
        for name, p in pii.items()
        if name != "any" and p >= PII_TYPE_THRESHOLD
    ]
    if pii["any"] >= pii_any_t and "any" not in pii_hits:
        pii_hits = ["any"] + pii_hits
    domain_hits, domain_top = present(domain, SIGNAL_META["topic.domain"]["class_thresholds"])
    op_hits, op_top = present(operation, SIGNAL_META["topic.operation"]["class_thresholds"])

    print(f"\n=== {title} ===")
    print(f"encoded tokens: {n_tokens} | PII length bucket: {bucket['name']} (any threshold {pii_any_t:.3f})")
    print(f"PII any: {pii['any']:.3f} | present: {pii_hits or '—'}")
    for name in PII_CLASSES:
        mark = " *" if (name == "any" and pii[name] >= pii_any_t) or (
            name != "any" and pii[name] >= PII_TYPE_THRESHOLD
        ) else ""
        print(f"  {name:<22} {pii[name]:.3f}{mark}")
    print(f"domain top: {domain_top} | present: {domain_hits or '—'}")
    for name in DOMAIN_CLASSES:
        t = SIGNAL_META["topic.domain"]["class_thresholds"][name]
        mark = " *" if domain[name] >= t else ""
        print(f"  {name:<24} {domain[name]:.3f}  (t={t}){mark}")
    print(f"operation top: {op_top} | present: {op_hits or '—'}")
    for name in OPERATION_CLASSES:
        t = SIGNAL_META["topic.operation"]["class_thresholds"][name]
        mark = " *" if operation[name] >= t else ""
        print(f"  {name:<24} {operation[name]:.3f}  (t={t}){mark}")
    status = SIGNAL_META["topic.domain"]["status"]
    low = [k for k, v in status.items() if v != "served"]
    if low:
        print("note: config marks some topic.domain classes low_support:", ", ".join(low))


report_signals("no PII / easy question", CONVERSATIONS[0]["messages"])
report_signals("coding agent", CONVERSATIONS[2]["messages"])

FICTIONAL_PII = [
    {
        "role": "user",
        "content": (
            "Please email the refund confirmation to Alice Example at alice.example@example.com "
            "or call +1-555-0100. Shipping address: 123 Example Street, Springfield. "
            "Date of birth on file: 1990-01-15."
        ),
    }
]
report_signals("fictional PII (example.com / 555)", FICTIONAL_PII)
print_vram("demo 2")


## 7. Demo 3 — a developer-defined question

The question head (`qh4in-full-e3`) fine-tunes layers 12–23 and adds a **bilinear pointer** (`q`, `k`, `log_scale`). The addon manifest is the serving contract used here:

- prefix = the same role blocks as routing
- suffix = `Question about the text above: {question}` + type header + option lines + the empty-think assistant tail
- query = last token of the full suffix
- option keys = last token of each option line (pieces tokenized separately, then concatenated)
- `logits = (k(options) @ q(query)) / sqrt(256) * exp(log_scale)` in fp32
- `softmax(logits / T[type])` with the stored temperatures (`noul` 1.20193, `choice` 1.03859, `score` 1.8292)

The public vllm-arc plugin does **not** expose this API yet; this cell applies those released tensors on the overlaid backbone. The hosted Decisions API is the production path.


In [ ]:
QH_RENDER = QH_MANIFEST["rendering"]["suffix"]
QH_T = QH_MANIFEST["calibration"]["temperatures"]
QH_WIDTH = float(QH_MANIFEST["pointer_head"]["width"])


def tokenize_piece(text):
    ids = TOKENIZER(text, add_special_tokens=False)["input_ids"]
    if not ids:
        raise ValueError(f"tokenizer produced no ids for {text!r}")
    return ids


def option_line(option):
    oid = str(option["id"])
    text = str(option.get("text") or "")
    if text == "" or text == oid:
        return f"{oid}\n"
    return f"{oid}: {text}\n"


def build_question_ids(messages, question, qtype, options):
    prefix = tokenize_piece(render_role_blocks(messages))
    header = QH_RENDER["headers"][qtype]
    user_open = QH_RENDER["user_open"].format(question=question, header=header)
    tail = QH_RENDER["tail"]
    pieces = [prefix, tokenize_piece(user_open)]
    option_last = []
    cursor = len(prefix) + len(pieces[1])
    for option in options:
        ids = tokenize_piece(option_line(option))
        cursor += len(ids)
        option_last.append(cursor - 1)
        pieces.append(ids)
    pieces.append(tokenize_piece(tail))
    all_ids = []
    for piece in pieces:
        all_ids.extend(piece)
    query_index = len(all_ids) - 1
    return all_ids, query_index, option_last


@torch.no_grad()
def ask(messages, question, qtype, options):
    ids, query_index, option_last = build_question_ids(messages, question, qtype, options)
    input_ids = torch.tensor([ids], device=DEVICE)
    out = forward_text(input_ids)
    if getattr(out, "last_hidden_state", None) is not None:
        hidden = out.last_hidden_state[0].float()
    else:
        hidden = TEXT.norm(out.hidden_states[-1][0].float())
    q = torch.nn.functional.linear(hidden[query_index], POINTER["q.weight"])
    k = torch.nn.functional.linear(hidden[option_last], POINTER["k.weight"])
    scale = POINTER["log_scale"].exp() / (QH_WIDTH ** 0.5)
    logits = (k @ q) * scale
    temp = float(QH_T[qtype])
    probs = torch.softmax(logits / temp, dim=0)
    ranked = sorted(
        (
            {
                "id": options[i]["id"],
                "text": options[i].get("text") or options[i]["id"],
                "logit": float(logits[i]),
                "prob": float(probs[i]),
            }
            for i in range(len(options))
        ),
        key=lambda r: -r["prob"],
    )
    return {
        "type": qtype,
        "question": question,
        "n_tokens": len(ids),
        "answer": ranked[0],
        "options": ranked,
    }


def print_question(title, result):
    print(f"\n=== {title} ===")
    print(f"{result['type']}: {result['question']}")
    print(f"encoded tokens: {result['n_tokens']}")
    for row in result["options"]:
        bar = "#" * int(round(row["prob"] * 20))
        print(f"  {row['prob']:5.1%}  {bar:<20}  {row['id']}: {row['text']}")
    print(f"top: {result['answer']['id']} ({result['answer']['prob']:.1%})")


NOUL_OPTIONS = [{"id": "true", "text": "true"}, {"id": "false", "text": "false"}]
CHOICE_OPTIONS = [
    {"id": "local_small", "text": "a cheap small model"},
    {"id": "mid", "text": "a mid-tier general model"},
    {"id": "strong", "text": "a strong frontier model"},
]

for case in CONVERSATIONS:
    noul = ask(
        case["messages"],
        "Does this request need a stronger model?",
        "noul",
        NOUL_OPTIONS,
    )
    choice = ask(
        case["messages"],
        "Which of these models should answer?",
        "choice",
        CHOICE_OPTIONS,
    )
    print_question(f"{case['name']} — need a stronger model?", noul)
    print_question(f"{case['name']} — which tier?", choice)

print_vram("demo 3")


## 8. Caveats and cleanup

**What this notebook is for.** A free-T4 walkthrough of ARC 1.0's released heads: Stage-1 success and usage on a small named pool, the Sage vs Swift utility policy from the card, PII / topic MLPs, and one bilinear question-head example.

**What it is not.** The hosted Routes / Generation / Decisions APIs ([docs.rayline.ai](https://docs.rayline.ai) — those need a Rayline key). Stage-2 reasoning-effort selection (weights are present; `vllm-arc` does not serve them yet). Bit-exact parity with the goldens, which were recorded on vLLM + GATHER readouts + `canonical_v2` serialization on an L4. Prefer-local hybrid routing. Calling any of the predicted models — this notebook only **scores** the conversation.

**API reminders (card / NOTICE / vllm-arc / question-head manifest).**
- Predictions are not a policy. You supply V, eligibility, and prices.
- `runtime/prices.json` is a reference card; override it with your rates.
- Encoder readout: depth-12 last token, then RMSNorm. Question-head readout: 24-layer `last_hidden_state`.
- Do not install `flash-attn` / `flash-linear-attention` / `causal-conv1d` on free Colab. Scores on the PyTorch fallback will not match a kernel-backed L4/H100 vLLM run to many decimals.
- PII labels are presence scores, not a compliance program. Keep a human in the loop for anything that ships.

**Licence.** Apache 2.0.

**Links again:** [model card](https://huggingface.co/rayline-ai/ARC-1.0) · [docs](https://docs.rayline.ai) · [vllm-arc](https://github.com/rayline-ai/vllm-arc) · [announcement](https://x.com/davidvgilmore/status/2120583960447029699)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup")
for name in ("model", "TEXT", "STAGE_ONE", "POINTER", "signal_tensors", "stage_tensors", "upper"):
    try:
        del globals()[name]
    except KeyError:
        pass
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("ARC 1.0 released. Re-run the load cells if you want another score.")
